# Two-Tower hard negatives ablation

Issue: #25

## Question

In-batch negatives are mostly easy: a random item from another category is trivially different from the positive. Does adding **hard negatives from the same category** as the positive make the retriever better at telling apart items inside a category, which is exactly what our queries need?

## Data used

The same data, catalog, training examples, seed, batch size and epoch budget as #24 (shared code in `notebooks/tt_setup.py`). Evaluation: the 56,544 ranking queries, exact in-category search.

Output: `data/models/two_tower_hard/` (model, config, item embeddings).

**Hard negatives.** For every training example, 4 items are drawn uniformly from the catalog items of the **same category** as the positive and used as extra negatives for that example. Two safeguards:
- an item that the same session actually found later (a relevant label) is masked out, because it would be a false negative;
- the logQ correction is applied to them too, with their sampling probability (`1 / category size`).

Both models use logQ and keep the epoch with the lowest validation loss, as in #24.

In [ ]:
import json
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import paired_bootstrap_diff
from tt_setup import hard_negatives, loss_with_hard, score, setup, train

DATA = Path("data") if Path("data").exists() else Path("../data")
ART = Path("artifacts") if Path("artifacts").exists() else Path("../artifacts")
OUT = DATA / "models" / "two_tower_hard"
OUT.mkdir(parents=True, exist_ok=True)
t0 = time.time()
ns = setup(DATA, ART)
print(f"setup in {time.time() - t0:.0f} s: catalog {ns.n_items:,} items, {ns.n_examples:,} training examples from {ns.n_train_queries:,} queries, evaluation queries {len(ns.prepared):,}")

## Checks

1. Every hard negative is in the same category as its positive.
2. A masked (false-negative) hard negative changes nothing: the loss equals the loss without that column.
3. With logits from a random model, the hard-negative loss is finite and differs from the in-batch loss.

In [ ]:
rng = np.random.default_rng(0)
pos = ns.item_of[:5000]
hn = hard_negatives(ns, pos, rng, per_row=4)
cats = ns.item_cat.numpy()
assert hn.shape == (5000, 4)
assert (cats[hn] == cats[pos][:, None]).all(), "a hard negative is in another category"
print(f"1. all {hn.size:,} sampled hard negatives are in the category of their positive")

torch.manual_seed(0)
B, d = 8, 16
q = torch.nn.functional.normalize(torch.randn(B, d), dim=-1)
i = torch.nn.functional.normalize(torch.randn(B, d), dim=-1)
hv = torch.nn.functional.normalize(torch.randn(B, 2, d), dim=-1)
items = torch.arange(B)
h_idx = torch.tensor([[100, 101]] * B)
lq = torch.zeros(200)
ok_all = torch.ones(B, 2, dtype=torch.bool)
ok_mask = ok_all.clone(); ok_mask[:, 1] = False
with_two = loss_with_hard(q, i, items, hv, h_idx, ok_all, lq, torch.zeros(B, 2))
with_one_masked = loss_with_hard(q, i, items, hv, h_idx, ok_mask, lq, torch.zeros(B, 2))
only_first = loss_with_hard(q, i, items, hv[:, :1], h_idx[:, :1], ok_all[:, :1], lq, torch.zeros(B, 1))
assert torch.isclose(with_one_masked, only_first, atol=1e-6), "a masked hard negative still changed the loss"
assert not torch.isclose(with_two, only_first)
print("2. a masked hard negative has no effect on the loss")
from twotower import sampled_softmax_loss

assert torch.isfinite(with_two) and not torch.isclose(with_two, sampled_softmax_loss(q, i, items))
print("3. hard-negative loss is finite and differs from the in-batch loss")

## Train both models with the same seed and budget

In [ ]:
base, hist_base, t_base = train(ns, hard=0, label="in-batch only  ")
hard, hist_hard, t_hard = train(ns, hard=4, label="+4 hard negs    ")

## Evaluate

In [ ]:
res_base, rec_base, s_base, vec_base, _ = score(ns, base, "in-batch only ")
res_hard, rec_hard, s_hard, vec_hard, _ = score(ns, hard, "+ hard negatives")
print(ns.prep_stats)

table = pl.DataFrame({
    "model": ["in-batch negatives only", "+ 4 same-category hard negatives"],
    "ndcg10": [s_base["ndcg10"][0], s_hard["ndcg10"][0]],
    "recall20": [rec_base[20], rec_hard[20]],
    "recall50": [rec_base[50], rec_hard[50]],
    "recall100": [rec_base[100], rec_hard[100]],
    "recall200": [rec_base[200], rec_hard[200]],
    "train_seconds": [round(t_base), round(t_hard)],
})
print(table)
j = res_hard.join(res_base.select("session", pl.col("ndcg10").alias("b")), on="session")
m, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["b"].to_numpy())
print(f"NDCG@10 hard minus in-batch: {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
torch.save(hard.state_dict(), OUT / "model.pt")
np.save(OUT / "item_embeddings.npy", vec_hard.astype(np.float32))
np.save(OUT / "catalog_aids.npy", ns.catalog_aids)
(OUT / "config.json").write_text(json.dumps({
    "hard_negatives_per_example": 4, "logq": True, "epochs": 4, "batch_size": 2048, "seed": 0,
    "train_seconds": {"in_batch": round(t_base), "hard": round(t_hard)},
    "recall": {"in_batch": rec_base, "hard": rec_hard},
}, indent=2, default=float))
print("saved to", OUT, "; peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

Both models use the same seed, batch size and epoch budget. On the 56,544 evaluation queries (exact in-category search):

| Model | NDCG@10 | recall@20 | recall@50 | recall@100 | recall@200 | train time |
|---|---|---|---|---|---|---|
| in-batch negatives only | 0.0458 | 0.1206 | 0.1975 | 0.2745 | 0.3667 | 255 s |
| + 4 same-category hard negatives | 0.0457 | 0.1207 | 0.1975 | 0.2745 | 0.3664 | 270 s |

Paired NDCG@10 difference (hard minus in-batch): **-0.0001 [-0.0002, +0.0001]**, i.e. no difference.

- **Hard negatives did not help here.** Recall is the same to three decimals at every cutoff and the interval on the NDCG@10 difference contains zero.
- **Decision: the in-batch-only model is used going forward** (#26). It is simpler and about as fast, and the hard-negative model gives no gain. The hard-negative model is saved next to it in `data/models/two_tower_hard/` for reference.
- **Why there is no effect.** Both runs keep epoch 1, where the model has barely started to separate items inside a category, and the validation loss rises from epoch 2 on (more steeply with hard negatives: 4.14 vs 3.94 at epoch 4), which is the same overfitting seen in #24. Harder negatives make the training task harder for the ID embeddings to memorise, but the kept epoch-1 model has not yet learned enough for that to matter. A regularised model that trains for longer might show a difference; that was not tried.
- The in-batch model reproduces #24 (recall@100 0.2745). Training times here are longer than the 82 s measured in #24 because the machine was running other work; only the comparison between the two rows is meaningful.

## What I learned

To be written by the owner of the project.